# Modeling DATA

Modeling data into one single model
provides more robust way to pass on information accross the module
Implement changes with no worries about the dataset states

Impute and load up one unified dataframe state with one unified tool

Features
1. Automated internal version controll flow for dataset
2. serving fetch and edit api gound with automated commiting and fetching latest changes
3. Request managers for incomming edit requests along with all fetch information retrives
4. Dedicated version fetching tools imbeded

**Versioning feature**
Leading with delta based versioning system
Building version with prior edits
Easy loading less storage

### NoteBook targets implementations for
1. Version controll with comparing delta and version build
2. request handler object prototype
3. Execution manager and history mechanism to work with versioning

In [1]:
# Loading testing dataset
%run clean.ipynb
df_test

Indexing Elements ['user_id']
String Columns : ['plan_type', 'device']
Numerical Columns : ['age', 'salary']
target plan_type
target device
{'user_id': 0, 'age': 0, 'salary': 0, 'plan_type': 0, 'device': 2}


,user_id,age,salary,plan_type,device
1,102,-12.0,62000.0,Premium,Desktop
2,103,250.0,75000.0,Basic,Mobile
4,105,58.0,-999.0,Standard,Desktop
5,106,31.0,99000000.0,Basic,Mobile
7,108,36.0,60000.0,Hacker,Tablet
8,109,36.0,68000.0,Basic,NULL
9,110,50.0,88000.0,Standard,None


## Version control for dataframes

1. Initiation and delta storage

Initiating with first data load, First header versioning into history
auto commiting for the base root to start with versioning

2. Storing and fetching

+ fetch changes in df
+ store the changes in history
+ build the new changed element


### Capturing the difference for delta
```DataFrame.index.difference```
We can extracts info about new additions
with difference api we can fetch how many indexing and columns have changed removed or addedd to the data frame with this change

In [2]:
# DF version control system

HISTORY = [] # with dict about information
DF =  df_test.copy(deep=True)  # Root target for versioning

def track_change(new_df : pd.DataFrame, message:str):
    '''
    1. capture changes
    2. build the delta object
    3. commit to history and update the version
    '''
    # First milestone of tracking changes occured into the dataset
    if not new_df.index.is_unique:
        raise ValueError("Can not load this data frame for delta build")

    # New Adddons with respect to source
    new_columns = new_df.columns.difference(DF.columns)
    new_rows = new_df.index.difference(DF.index)

    # New removals
    removed_columns = DF.columns.difference(new_df.columns)
    removed_rows = DF.index.difference(new_df.index)

    existing_columns = new_df.columns.intersection(DF.columns)
    existing_rows = new_df.index.intersection(DF.index)
    
    print(f'''
    New additions
    colum : {new_columns}
    rows : {new_rows}

    Removals
    column: {removed_columns}
    rows : {removed_rows}

    Existing Internal changes
    columns : {existing_columns}
    rows : {existing_rows}
    ''')

# Weird edited source columns
new_df = DF.copy(deep=True)

print(new_df.head(2))

new_df['pansioning'] = DF['age'] * 10
new_df.loc[10] = [1, 111, 1500.0, 50000.0, 'halwa', 'satelite']

#new_df = new_df.drop('age', axis=1)
new_df = new_df.drop(2)

new_df['user names'] = [ f'user{i}' for i in range(1, 8)]

new_df.loc[1, 'user names']='new name'

new_df.iloc[1,1] = 10

track_change(new_df, 'Made some non-required changes')

DF, new_df

   user_id    age   salary plan_type   device
1      102  -12.0  62000.0   Premium  Desktop
2      103  250.0  75000.0     Basic   Mobile

    New additions
    colum : Index(['pansioning', 'user names'], dtype='str')
    rows : Index([10], dtype='int64')

    Removals
    column: Index([], dtype='str')
    rows : Index([2], dtype='int64')

    Existing Internal changes
    columns : Index(['user_id', 'age', 'salary', 'plan_type', 'device'], dtype='str')
    rows : Index([1, 4, 5, 7, 8, 9], dtype='int64')
    


(   user_id    age      salary plan_type   device
 1      102  -12.0     62000.0   Premium  Desktop
 2      103  250.0     75000.0     Basic   Mobile
 4      105   58.0      -999.0  Standard  Desktop
 5      106   31.0  99000000.0     Basic   Mobile
 7      108   36.0     60000.0    Hacker   Tablet
 8      109   36.0     68000.0     Basic     NULL
 9      110   50.0     88000.0  Standard     None,
     user_id    age      salary plan_type   device pansioning user names
 1       102  -12.0     62000.0   Premium  Desktop     -120.0   new name
 4       105   10.0      -999.0  Standard  Desktop      580.0      user2
 5       106   31.0  99000000.0     Basic   Mobile      310.0      user3
 7       108   36.0     60000.0    Hacker   Tablet      360.0      user4
 8       109   36.0     68000.0     Basic     NULL      360.0      user5
 9       110   50.0     88000.0  Standard     None      500.0      user6
 10        1  111.0      1500.0   50000.0    halwa   satelite      user7)